<br/>
<img src="images/utfsm.png" alt="" width="130px" align="left"/>
<img src="images/utfsm.png" alt="" width="130px" align="right"/>
<div align="center">
<h1>Redes Generativas</h1>
<br/><br/>
Dr. Nicolás Gálvez Ramírez<br/>
Dr. Patricio Olivares Roncagliolo<br/><br/>
Ingeniería Civil Telemática<br/>
Departamento de Eléctronica<br/>
Universidad Técnica Federico Santa María
</div>
<br>
Fuentes: 
<br>
"Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"

## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** el entrenamiento adversarial: generador frente a discriminador.
2. **Implementar** una GAN simple y reconocer la arquitectura de una DCGAN.
3. **Entender** los Autoencoders (AE) y los Variational Autoencoders (VAE): cuello de botella, reparametrización y ELBO.
4. **Generar** e interpolar muestras sintéticas en el espacio latente.
5. **Reconocer** problemas de entrenamiento (*mode collapse*) y métricas de evaluación (FID, IS).

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb).

## 1. Modelos Generativos

Un **modelo generativo** aprende la *distribución de probabilidad* $p(x)$ de un conjunto de datos para poder **sintetizar muestras nuevas** parecidas a las originales. Esto contrasta con los modelos *discriminativos*, que solo aprenden la frontera $p(y \mid x)$ entre clases.

La idea central es el **espacio latente**: una representación comprimida y de baja dimensión $z$ desde la cual se pueden *decodificar* muestras del espacio de datos $x$. Recorrer o muestrear este espacio latente permite generar contenido nuevo.

### Conceptos clave
- **Espacio latente** ($z$): representación compacta que captura los factores de variación de los datos.
- **Distribución de datos** ($p(x)$): la densidad real (desconocida) que el modelo intenta aproximar.
- **Muestreo**: extraer $z \sim p(z)$ y decodificarlo para obtener una muestra sintética $\hat{x}$.

En esta clase estudiamos tres familias de modelos generativos: **Autoencoders (AE)**, **Variational Autoencoders (VAE)** y **Generative Adversarial Networks (GAN)**.

## 2. Autoencoders (AE)

Un **autoencoder (AE)** es una red neuronal entrenada para **reproducir su entrada en su salida**, forzando la información a pasar por una representación interna comprimida. Al restringir el flujo a través de un **cuello de botella** (*bottleneck*) de baja dimensión, la red aprende a conservar solo los factores más relevantes de los datos.

Consta de dos partes:
- **Codificador** (*encoder*): comprime la entrada $x$ en un código latente $z = f(x)$ de menor dimensión.
- **Decodificador** (*decoder*): reconstruye la entrada a partir del código, $\hat{x} = g(z)$.

El objetivo es **minimizar el error de reconstrucción** entre la entrada y su reconstrucción, por ejemplo con la pérdida $\mathcal{L}(x, \hat{x}) = \lVert x - \hat{x} \rVert^2$. Se usan sobre todo para **reducción de dimensionalidad** y **extracción de características**.

<img src="images/AE.png" alt="Arquitectura de un autoencoder: encoder, cuello de botella y decoder" width="800px" align="center"/>

Fuente: "Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"

<img src="images/AE2.png" alt="Encoder y decoder de un autoencoder" width="800px" align="center"/>

Fuente: "https://towardsdatascience.com/applied-deep-learning-part-3-autoencoders-1c083af4d798"

### Aplicaciones de los Autoencoders
- Reducción de dimensionalidad (compresión)
- Eliminación de ruido en imágenes (*denoising*)
- Detección de anomalías

<img src="images/AE_noise.png" alt="Autoencoder para eliminación de ruido: entrada con ruido, reconstrucción limpia" width="800px" align="center"/>

*Ejemplo de eliminación de ruido (denoising): el AE recibe una imagen con ruido y aprende a reconstruir la versión limpia.*

In [ ]:
# ─── Autoencoder apilado (stacked AE) sobre Fashion-MNIST ───
import tensorflow as tf
# Ejemplo usando fashion mnist
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Cost",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"] # 10 clases

X_train_full = X_train_full/255.0 # Escalamiento

from sklearn.model_selection import train_test_split
# Utilización de datos de validación y entrenamiento del modelo
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, 
                                                  stratify=y_train_full)

stacked_encoder = tf.keras.models.Sequential([
tf.keras.layers.Flatten(input_shape=[28, 28]),
tf.keras.layers.Dense(100, activation="selu"),
tf.keras.layers.Dense(30, activation="selu"),
])

stacked_decoder = tf.keras.models.Sequential([
tf.keras.layers.Dense(100, activation="selu", input_shape=[30]),
tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

stacked_ae = tf.keras.models.Sequential([stacked_encoder, stacked_decoder])

stacked_ae.compile(loss="binary_crossentropy",
optimizer=tf.keras.optimizers.SGD(lr=1.5))
history = stacked_ae.fit(X_train, X_train, epochs=50, validation_data=[X_val, X_val])

In [ ]:
# ─── Comparación entrada original vs. reconstrucción (stacked AE) ───
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(15,15))
prediction = stacked_ae.predict(X_train)

i=19

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(X_train[i], cmap='gist_yarg')
ax1.set_xlabel("Original")
ax1.set_title(class_names[y_train[i]])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(prediction[i], cmap='gist_yarg')
ax2.set_xlabel("Reconstrucción")
ax2.set_title(class_names[y_train[i]])


plt.show()

## 3. Autoencoder Convolucional

Cuando trabajamos con imágenes conviene reemplazar las capas densas por **capas convolucionales**, que explotan la estructura espacial y comparten parámetros, capturando patrones locales de forma más eficiente.

- El **encoder** usa `Conv2D` + `MaxPooling` para reducir progresivamente la resolución espacial y aumentar la profundidad de canales.
- El **decoder** usa **convoluciones transpuestas** (`Conv2DTranspose`, a veces llamadas *deconvoluciones*), que **aumentan** la resolución (*upsampling*) hasta recuperar el tamaño original de la imagen.

> 💡 Estas mismas convoluciones transpuestas son la base del **generador de una DCGAN** (*Deep Convolutional GAN*), que veremos como la variante convolucional de las GAN.

In [ ]:
# ─── Autoencoder convolucional: encoder (Conv2D) + decoder (Conv2DTranspose) ───
conv_encoder = tf.keras.models.Sequential([
tf.keras.layers.Reshape([28, 28, 1], input_shape=[28, 28]),
tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2),
tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2),
tf.keras.layers.Conv2D(64, kernel_size=3, padding="same", activation="selu"),
tf.keras.layers.MaxPool2D(pool_size=2)
])
conv_decoder = tf.keras.models.Sequential([
tf.keras.layers.Conv2DTranspose(32, kernel_size=3, strides=2, padding="valid",activation="selu", input_shape=[3, 3, 64]),
tf.keras.layers.Conv2DTranspose(16, kernel_size=3, strides=2, padding="same",
activation="selu"),
tf.keras.layers.Conv2DTranspose(1, kernel_size=3, strides=2, padding="same",
activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

conv_ae = tf.keras.models.Sequential([conv_encoder, conv_decoder])

conv_ae.compile(loss="binary_crossentropy",
optimizer=tf.keras.optimizers.SGD(lr=1.5))
history = conv_ae.fit(X_train, X_train, epochs=50, validation_data=[X_val, X_val])

In [ ]:
# ─── Comparación entrada original vs. reconstrucción (autoencoder convolucional) ───
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(15,15))
prediction = conv_ae.predict(X_train)

i=7

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(X_train[i], cmap='gist_yarg')
ax1.set_xlabel("Original")
ax1.set_title(class_names[y_train[i]])
ax2 = fig.add_subplot(1,2,2)
ax2.imshow(prediction[i], cmap='gist_yarg')
ax2.set_xlabel("Reconstrucción")
ax2.set_title(class_names[y_train[i]])

plt.show()

## 4. Variational Autoencoders (VAE)

Un **Variational Autoencoder (VAE)** es una variante de los autoencoders que combina la reducción de dimensionalidad con un **modelo generativo probabilístico**.

Un autoencoder tradicional tiene:
- Un **codificador** que reduce la dimensionalidad de los datos de entrada a una representación latente.
- Un **decodificador** que reconstruye los datos originales a partir de esa representación latente.

En un VAE también existen codificador y decodificador, pero se introduce un **componente probabilístico**: en lugar de un único código $z$, el encoder produce una **distribución** sobre el espacio latente (una media $\mu$ y una varianza $\sigma^2$) desde la cual se **muestrea** $z$.

<img src="images/VAE_Basic.png" alt="Esquema de un Variational Autoencoder con muestreo del espacio latente" width="800px" align="center"/>

Fuente: "https://en.wikipedia.org/wiki/Variational_autoencoder"

<img src="images/VAE2.png" alt="Espacio latente continuo aprendido por un VAE" width="800px" align="center"/>

Fuente: "https://towardsdatascience.com/intuitively-understanding-variational-autoencoders-1bfe67eb5daf"

### Truco de reparametrización

Para poder **entrenar con backpropagation** a través de un paso de muestreo aleatorio, el VAE no muestrea $z$ directamente de $\mathcal{N}(\mu, \sigma^2)$, sino que lo reescribe como una transformación determinista de un ruido independiente:

$$z = \mu + \sigma \odot \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

Así el gradiente puede fluir hacia $\mu$ y $\sigma$, mientras el ruido $\epsilon$ queda fuera del camino de los parámetros.

### Función de pérdida: ELBO

El VAE maximiza el **ELBO** (*Evidence Lower Bound*), lo que equivale a minimizar:

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(z \mid x)}\big[-\log p(x \mid z)\big]}_{\text{reconstrucción}} + \underbrace{D_{KL}\big(q(z \mid x)\,\Vert\,p(z)\big)}_{\text{regularización}}$$

- El **término de reconstrucción** empuja a que el decoder reproduzca la entrada.
- El **término KL** (divergencia de Kullback–Leibler) acerca la distribución latente $q(z \mid x)$ a una **normal estándar** $p(z) = \mathcal{N}(0, I)$, produciendo un **espacio latente continuo y suave**, apto para generación e interpolación.

In [ ]:
# ─── Encoder y decoder variacionales (capa Sampling = truco de reparametrización) ───
import tensorflow as tf
class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        return tf.keras.backend.random_normal(tf.shape(log_var)) * tf.keras.backend.exp(log_var / 2) + mean

codings_size = 10
inputs = tf.keras.layers.Input(shape=[28, 28])
z = tf.keras.layers.Flatten()(inputs)
z = tf.keras.layers.Dense(150, activation="selu")(z)
z = tf.keras.layers.Dense(100, activation="selu")(z)
codings_mean = tf.keras.layers.Dense(codings_size)(z) # μ
codings_log_var = tf.keras.layers.Dense(codings_size)(z) # γ
codings = Sampling()([codings_mean, codings_log_var])
variational_encoder = tf.keras.Model(
inputs=[inputs], outputs=[codings_mean, codings_log_var, codings])

decoder_inputs = tf.keras.layers.Input(shape=[codings_size])
x = tf.keras.layers.Dense(100, activation="selu")(decoder_inputs)
x = tf.keras.layers.Dense(150, activation="selu")(x)
x = tf.keras.layers.Dense(28 * 28, activation="sigmoid")(x)
outputs = tf.keras.layers.Reshape([28, 28])(x)
variational_decoder = tf.keras.Model(inputs=[decoder_inputs], outputs=[outputs])

In [ ]:
# ─── Modelo VAE completo: pérdida de reconstrucción + término KL (ELBO) ───
_, _, codings = variational_encoder(inputs)
reconstructions = variational_decoder(codings)
variational_ae = tf.keras.Model(inputs=[inputs], outputs=[reconstructions])

latent_loss = -0.5 * tf.keras.backend.sum(
1 + codings_log_var - tf.keras.backend.exp(codings_log_var) - tf.keras.backend.square(codings_mean),
axis=-1)
variational_ae.add_loss(tf.keras.backend.mean(latent_loss) / 784.)
variational_ae.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [ ]:
# ─── Entrenamiento del VAE ───
history = variational_ae.fit(X_train, X_train, epochs=50, batch_size=128,
validation_data=[X_val, X_val])

In [ ]:
# ─── Generación de nuevas muestras desde el espacio latente del VAE ───
codings = tf.random.normal(shape=[12, codings_size])
images = variational_decoder(codings).numpy()

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(3,3))
prediction = conv_ae.predict(X_train)

i=8

ax1 = fig.add_subplot(1,2,1)
ax1.imshow(images[i], cmap='gist_yarg')
ax1.set_title("Muestra generada (VAE)")
plt.show()

En los VAE, el codificador crea instancias a partir del muestreo de las variables latentes, lo que habilita a estos modelos para **generar datos distintos** de los que se utilizaron en su entrenamiento inicial.

Gracias a que el espacio latente es **continuo**, también podemos **interpolar** entre dos códigos $z_1$ y $z_2$ (por ejemplo $z = (1-\alpha)\,z_1 + \alpha\,z_2$, con $\alpha \in [0, 1]$) y decodificar cada punto intermedio para obtener una **transición suave** entre muestras.

## 5. Generative Adversarial Networks (GAN)

Las **Redes Generativas Adversarias** (GAN, por sus siglas en inglés) son una arquitectura de red neuronal usada para **generar datos nuevos**. Se basan en un entrenamiento **adversarial** entre dos redes que compiten entre sí:

- **Generador** ($G$): crea datos nuevos a partir de un vector de **ruido aleatorio** $z$ tomado del espacio latente.
- **Discriminador** ($D$): evalúa si un dato es **real** (proveniente del conjunto de entrenamiento) o **falso** (generado por $G$).

El generador busca "engañar" al discriminador, mientras el discriminador busca no dejarse engañar; ambos mejoran de forma conjunta.

<img src="images/GAN.png" alt="Arquitectura de una GAN: generador, discriminador y flujo de datos reales y falsos" width="800px" align="center"/>

Fuente: "https://www.microsoft.com/en-us/research/blog/how-can-generative-adversarial-networks-learn-real-life-distributions-easily/"

### Entrenamiento de una red GAN

El entrenamiento consta de dos etapas que se **alternan**:
- **Entrenamiento del discriminador**: alimentamos al generador con ruido gaussiano para producir imágenes falsas y completamos el lote concatenando un número igual de imágenes reales. Las etiquetas se fijan en 0 para las imágenes falsas y 1 para las reales, y entrenamos al discriminador con este lote.
- **Entrenamiento del generador**: suministramos ruido gaussiano al modelo GAN completo. El generador produce imágenes falsas y el discriminador intenta clasificarlas. Queremos que el discriminador *crea* que las imágenes falsas son reales, por lo que las etiquetas se fijan en 1 (el discriminador permanece congelado en esta fase).

### Función de pérdida minimax

El generador $G$ y el discriminador $D$ juegan un **juego de suma cero**: $D$ intenta maximizar su capacidad de distinguir real de falso, mientras $G$ intenta minimizarla. Esto se expresa como un problema **minimax**:

$$\min_G \max_D V(D, G) = \mathbb{E}_{x \sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

### Mode collapse y otros retos

Un problema típico de las GAN es el **colapso de modos** (*mode collapse*): el generador produce muy poca variedad (por ejemplo, siempre la misma imagen) porque halla unas pocas muestras que engañan al discriminador. Otros retos son la **inestabilidad** del entrenamiento y los **gradientes que se desvanecen** cuando el discriminador se vuelve demasiado bueno.

### DCGAN (Deep Convolutional GAN)

La **DCGAN** reemplaza las capas densas por **convoluciones**: el discriminador usa `Conv2D` con *stride* y el generador usa **convoluciones transpuestas** (`Conv2DTranspose`) para transformar el vector de ruido latente en una imagen. Esta variante mejora notablemente la calidad y la estabilidad frente a la GAN totalmente conectada.

In [ ]:
# ─── Definición del generador y el discriminador de la GAN ───
import tensorflow as tf
codings_size = 30
generator = tf.keras.models.Sequential([
tf.keras.layers.Dense(100, activation="selu", input_shape=[codings_size]),
tf.keras.layers.Dense(150, activation="selu"),
tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
tf.keras.layers.Reshape([28, 28])
])

discriminator = tf.keras.models.Sequential([
tf.keras.layers.Flatten(input_shape=[28, 28]),
tf.keras.layers.Dense(150, activation="selu"),
tf.keras.layers.Dense(100, activation="selu"),
tf.keras.layers.Dense(1, activation="sigmoid")
])
gan = tf.keras.models.Sequential([generator, discriminator])

discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

In [ ]:
# ─── Entrenamiento adversarial alternado (discriminador → generador) ───
from tqdm import tqdm
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=50):
    generator, discriminator = gan.layers
    for epoch in tqdm(range(n_epochs)):
        for X_batch in dataset:
            # phase 1 - training the discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = tf.cast(generator(noise),tf.float32)
            X_batch = tf.cast(X_batch, tf.float32) 
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            discriminator.trainable = True # Acá el discriminador se marca como entrenable
            discriminator.train_on_batch(X_fake_and_real, y1)
            # phase 2 - training the generator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)
train_gan(gan, dataset, batch_size, codings_size)

In [ ]:
# ─── Generación de una muestra con el generador entrenado ───
noise = tf.random.normal(shape=[1, codings_size])
image = generator(noise).numpy().reshape((28,28))

import matplotlib.pyplot as plt
fig = plt.figure(figsize=(3,2))

ax1 = fig.add_subplot(1,1,1)
ax1.imshow(image, cmap='gist_yarg')
ax1.set_title("Muestra generada (GAN)")
plt.show()

## 6. Evaluación de Modelos Generativos

Evaluar modelos generativos es difícil porque no existe una única "respuesta correcta": una buena muestra es simplemente *verosímil*, no idéntica a un dato real. Dos métricas estándar para imágenes son:

- **Inception Score (IS)**: usa una red *Inception* preentrenada para medir que cada muestra sea *nítida* (clasificable con confianza) y que el conjunto sea *diverso*. **Mayor es mejor**.
- **Fréchet Inception Distance (FID)**: compara las estadísticas (media y covarianza) de las activaciones *Inception* de las imágenes reales y las generadas. **Menor es mejor** (0 = distribuciones idénticas). Hoy es la métrica más usada por correlacionar mejor con el juicio humano.

## 📌 Resumen

| Modelo | Idea central | Espacio latente | Uso principal |
|--------|--------------|-----------------|----------------|
| **Autoencoder (AE)** | Comprimir y reconstruir minimizando el error | Determinista, sin estructura garantizada | Reducción de dimensionalidad, denoising, anomalías |
| **AE Convolucional** | AE con `Conv2D` / `Conv2DTranspose` | Determinista, espacial | Reconstrucción de imágenes |
| **VAE** | AE probabilístico entrenado con el ELBO | Continuo y regular ($\mu, \sigma$) | Generación e interpolación |
| **GAN** | Juego minimax generador vs. discriminador | Ruido $z$ muestreado | Generación de muestras realistas |

### Conceptos clave
- **Espacio latente**: representación comprimida desde la que se decodifican muestras.
- **Reparametrización** ($z = \mu + \sigma \odot \epsilon$): permite entrenar el muestreo del VAE con backpropagation.
- **Minimax** ($\min_G \max_D V(D,G)$): objetivo adversarial de la GAN.
- **Mode collapse**: falta de diversidad en las muestras generadas por una GAN.
- **Métricas**: FID (menor es mejor) e IS (mayor es mejor).

## 🔗 Próximos pasos
- Módulo siguiente: [05-4 Arquitecturas CNN Avanzadas](../05-4_AdvancedCNNArchitectures/).
- Lecturas recomendadas: ver la sección de Referencias.

## 📚 Referencias

- **Goodfellow, I. et al. (2014).** *Generative Adversarial Nets*. NeurIPS. [arXiv:1406.2661](https://arxiv.org/abs/1406.2661)
- **Kingma, D. P. & Welling, M. (2014).** *Auto-Encoding Variational Bayes*. ICLR. [arXiv:1312.6114](https://arxiv.org/abs/1312.6114)
- **Radford, A. et al. (2016).** *Unsupervised Representation Learning with Deep Convolutional GANs (DCGAN)*. ICLR. [arXiv:1511.06434](https://arxiv.org/abs/1511.06434)
- **Heusel, M. et al. (2017).** *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium (FID)*. NeurIPS. [arXiv:1706.08500](https://arxiv.org/abs/1706.08500)
- **Géron, A. (2019).** *Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow* (2.ª ed.), cap. 17. O'Reilly.
- **Keras.** Documentación oficial: [https://keras.io](https://keras.io)
- **TensorFlow.** Tutoriales de modelos generativos: [https://www.tensorflow.org/tutorials/generative](https://www.tensorflow.org/tutorials/generative)

In [ ]:
# ─── Utilidad de presentación: centra las imágenes de matplotlib en las diapositivas ───
from IPython.core.display import HTML
HTML("""
<style>
.output_png {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")
#codigo extra, para que imagenes de matplotlib
#estén centradas en las diapositivas, ejecutar antes de lanzar los ejemplos.